# Visualization Guide

TMAP visualizations are built with `TmapViz`. The same class handles
generic data, molecules (SMILES), images, and proteins — you just call
a different `add_*` method and the appropriate rendering is enabled.

| Data type | Method | Tooltip feature |
|-----------|--------|-----------------|
| Generic | `add_color_layout()` / `add_label()` | Text labels + color switching |
| Molecules | `add_smiles(smiles_list)` | 2D structure via SmilesDrawer |
| Images | `add_images(data_uri_list)` | Thumbnail preview |
| Proteins | `add_protein_ids(uniprot_ids)` | Mol* 3D viewer + UniProt metadata |

All produce self-contained HTML files with binary encoding (gzip-compressed
typed arrays). For very large datasets (1M+), use `viz.serve()` or `viz.write_static()`.

Every visualization includes a bottom toolbar with color, filter, search panels,
edge toggle, and dark/light theme toggle.

This notebook shows each workflow with minimal examples.

## Setup: fit a quick TMAP

We'll reuse the same model for all examples.

In [2]:
import numpy as np
from tmap import TMAP

# Synthetic binary fingerprints (3 clusters)
rng = np.random.default_rng(42)
n, n_bits = 50_000, 2048
fps = np.zeros((n, n_bits), dtype=np.uint8)
labels = []
for i in range(n):
    c = i % 3
    labels.append(f"Cluster {c}")
    base = rng.choice(400, 50, replace=False) + c * 500
    fps[i, base] = 1
    fps[i, rng.choice(n_bits, 20, replace=False)] = 1

model = TMAP(n_neighbors=20, seed=42).fit(fps)
print(f"Embedding: {model.embedding_.shape}, edges: {model.tree_.edges.shape[0]}")


Embedding: (50000, 2), edges: 49999


---
## 1. Generic data (base template)

The default workflow: add color layers and text labels. No special
dependencies. The dropdown in the HTML lets the user switch between colors.

In [3]:
viz = model.to_tmapviz()
viz.title = "Generic Example"
viz.background_color = "#FFFFFF"

# Categorical color layer
viz.add_color_layout("Cluster", labels, categorical=True, color="Set1")

# Continuous color layer
values = rng.uniform(0, 100, n)
viz.add_color_layout("Score", values.tolist(), categorical=False, color="viridis")

# Text labels (shown in tooltip)
viz.add_label("ID", [f"Point {i}" for i in range(n)])

# Export HTML
path = viz.write_html("generic_tmap.html")
print(f"Saved: {path}")

# Or show inline in Jupyter
viz.show()


Saved: generic_tmap.html


/Users/afloresep/work/tmap2/src/tmap/visualization/tmapviz.py:1012: UserWarning: Edges are not supported in notebook mode yet and will be ignored.
  scatter = self.to_widget(width=width, height=height, controls=controls)


---
## 2. Molecules with SMILES rendering

Call `add_smiles(smiles_list)` and the template auto-switches to
`smiles.html.j2`. Hovering over a point renders the 2D structure
via [SmilesDrawer](https://github.com/reymond-group/smilesDrawer).

**Requirements:** `pip install rdkit` (for fingerprints/properties, not for rendering)

In [4]:
from tmap import TMAP
from tmap.utils import fingerprints_from_smiles, molecular_properties
import pandas as pd

# Load molecules
smiles_df = pd.read_csv("../examples/cluster_65053.csv")
smiles = smiles_df["smiles"].tolist()

# Fingerprints + properties
fps, valid_idx = fingerprints_from_smiles(smiles, fp_type="morgan", radius=2, n_bits=2048)
smiles = [smiles[i] for i in valid_idx]
props = molecular_properties(smiles)

# Fit
mol_model = TMAP(n_neighbors=20, seed=42).fit(fps)

# Visualize
viz = mol_model.to_tmapviz()
viz.title = "Chemical Space"

viz.add_smiles(smiles)                 # <-- enables SmilesDrawer in tooltip
viz.add_color_layout("MW", props["mw"].tolist(), color="viridis")
viz.add_color_layout("LogP", props["logp"].tolist(), color="plasma")
viz.add_color_layout("Rings", props["n_rings"].tolist(), categorical=True, color="tab10")

path = viz.write_static("smiles_tmap")
print(f"Saved: {path}")


  [FP] 64,098/64,098 valid
  [Props] 64,098 done, 0 invalid
Saved: smiles_tmap


The HTML loads SmilesDrawer from a CDN. No server needed, just open the file.
Pinned cards also show the structure.

---
## 3. Image thumbnails

Call `add_images(data_uri_list)` to show image thumbnails on hover.
Each value must be a base64 data URI (`"data:image/png;base64,..."`).

Common workflow: encode images as embeddings (CLIP, DINOv2, etc.),
fit TMAP on the embeddings, then attach the original images for tooltips.

In [5]:
import base64
import io
from PIL import Image
import numpy as np

def image_to_data_uri(arr: np.ndarray, size: int = 64) -> str:
    """Convert a numpy array (H, W) or (H, W, C) to a data URI."""
    img = Image.fromarray(arr.astype(np.uint8))
    img = img.resize((size, size), Image.NEAREST)
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return f"data:image/png;base64,{b64}"


# Generate synthetic 8x8 grayscale "images" for demo
n_images = 200
image_data = rng.integers(0, 255, (n_images, 8, 8), dtype=np.uint8)
data_uris = [image_to_data_uri(img) for img in image_data]

# Use pixel means as features (in practice: CLIP/DINOv2 embeddings)
features = image_data.reshape(n_images, -1).astype(np.float32)

img_model = TMAP(metric="cosine", n_neighbors=10, seed=42).fit(features)

viz = img_model.to_tmapviz()
viz.title = "Image TMAP"
viz.add_color_layout("brightness", features.mean(axis=1).tolist())
viz.add_images(data_uris, tooltip_size=128)  # <-- enables image tooltip

path = viz.write_html("images_tmap.html")
print(f"Saved: {path}")


Saved: images_tmap.html


**Tip:** For real datasets (MNIST, Oxford Pets, etc.), encode images as
data URIs at a small size (64-128px). The `tooltip_size` parameter controls
display size. See `examples/pet_breed_audit.py` for a full example with
DINOv2 embeddings.

```python
# Real-world pattern:
embeddings = extract_embeddings(images, model="dinov2")   # (N, 768)
data_uris = [image_to_data_uri(img, size=64) for img in images]

tmap_model = TMAP(metric="cosine").fit(embeddings)
viz = tmap_model.to_tmapviz()
viz.add_images(data_uris, tooltip_size=128)
viz.add_color_layout("class", class_labels, categorical=True)
viz.write_html("image_explorer.html")
```

---
## 4. Proteins with 3D structure viewer

Call `add_protein_ids(uniprot_ids)` to enable:
- Clickable UniProt links in tooltips
- Mol* 3D structure viewer in pinned cards (lazy-loaded from AlphaFold DB)
- On-demand UniProt metadata (protein name, organism, gene)

See `notebooks/09_protein_analysis.ipynb` for the full workflow including
`fetch_alphafold` and `fetch_uniprot`.

In [6]:
# Minimal protein example (no network calls needed to build the TMAP,
# Mol* fetches structures on demand when you click a point)

# Suppose you have embeddings + UniProt IDs
uniprot_ids = np.load('/Users/afloresep/work/tmap2/data/ids_50k.npy')
prot_features = np.load('/Users/afloresep/work/tmap2/data/embeddings_50k.npy')
# Dummy embeddings for demo (in practice: ESM-2, ProtTrans, etc.)
print(uniprot_ids[0], prot_features[0])

prot_model = TMAP(metric="euclidean", n_neighbors=20, seed=42).fit(prot_features)

viz = prot_model.to_tmapviz()
viz.title = "Protein TMAP"
viz.add_protein_ids(uniprot_ids)  # <-- enables Mol* viewer + UniProt links
viz.add_label("ID", uniprot_ids)
viz.point_size= 3.0
path = viz.write_html("protiens_tmap.html")
print(f"Saved: {path}")


A0A6A4IZ81 [ 0.03013611 -0.03250122  0.00200462 ... -0.02760315 -0.01782227
  0.07147217]
Saved: protiens_tmap.html


**Full protein workflow** (from `09_protein_analysis.ipynb`):

```python
from tmap import TMAP
from tmap.utils import fetch_alphafold, fetch_uniprot

model = TMAP(metric="cosine").fit(embeddings)
viz = model.to_tmapviz()

# Structural metrics as color layers
af = fetch_alphafold(ids)  # pLDDT, disorder, etc.
viz.add_metadata(af)

# UniProt text annotations as tooltip labels
up = fetch_uniprot(ids)
viz.add_label("Protein", list(up["protein_name"]))
viz.add_label("Organism", list(up["organism_name"]))

# 3D structure viewer in cards
viz.add_protein_ids(ids)

viz.write_html("protein_explorer.html")
```

---
## 5. Advanced: filtering, search, and card configuration

For large exploratory datasets you can configure filtering, search, and
card layout declaratively.

In [9]:
viz = model.to_tmapviz()
viz.title = "Advanced Config"
viz.add_color_layout("Cluster", labels, categorical=True, color="Set1")
viz.add_color_layout("Score", rng.uniform(0, 100, n).tolist())
viz.add_label("ID", [f"Point {i}" for i in range(n)])
viz.add_label("bla", [f"bla {i}" for i in range(n)])

# Declare which columns appear in the filter panel
viz.filterable = ["Cluster"]

# Declare which columns are searchable
viz.searchable = ["ID", "bla"]

# Configure the pinned-card layout
viz.configure_card(
    title="bla",
    fields=["ID", "Score"],
)

# Per-column UI hints
viz.configure_column("ID", copyable=True)

path = viz.write_html("advanced_tmap.html")
print(f"Saved: {path}")


Saved: advanced_tmap.html


---
## 6. Large datasets: serve mode

For 1M+ points, a single HTML file gets large. Use `serve()` for a
local dev server or `write_static()` to produce files for hosting.

```python
# Local dev server (columns loaded lazily via HTTP fetch)
viz.serve(port=8050)

# Static files for nginx / S3 / GitHub Pages
viz.write_static("dist/my_tmap/")
```

---
## Output options summary

| Method | Output | Best for |
|--------|--------|---------|
| `viz.to_html()` | HTML string | Embedding in apps |
| `viz.write_html(path)` | Single `.html` file | Sharing, < 1M points |
| `viz.write_static(dir)` | Directory of files | Static hosting, 1M+ points |
| `viz.serve(port)` | Local HTTP server | Development, 1M+ points |
| `viz.show()` | Jupyter widget | Interactive exploration |
| `viz.to_widget()` | jscatter object | Programmatic widget access |
| `model.plot()` | Quick Jupyter plot | Fast preview |
| `model.plot_static()` | Matplotlib figure | Papers, static output |

## Quick reference

```python
from tmap import TMAP
model = TMAP().fit(data)
viz = model.to_tmapviz()

# --- Data type specific ---
viz.add_smiles(smiles_list)              # molecules
viz.add_images(data_uri_list)            # images
viz.add_protein_ids(uniprot_ids)         # proteins

# --- Always available ---
viz.add_color_layout(name, values)       # color layer
viz.add_label(name, values)              # tooltip text
viz.add_metadata(dict_or_df)             # bulk add
viz.set_edges(s, t)                      # tree edges

# --- UI configuration ---
viz.filterable = ["column_name"]         # filter panel
viz.searchable = ["column_name"]         # search box
viz.configure_card(title=..., fields=...)# pinned card layout
viz.configure_column(name, copyable=True)# per-column hints

# --- Output ---
viz.write_html("output.html")
viz.show()  # Jupyter
```